# 01 — Hashing et HMAC

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :
- expliquer ce qu'est une fonction de hachage et ses propriétés ;
- utiliser `hashlib` pour calculer des empreintes (SHA-256, SHA-3, BLAKE2) ;
- calculer le hash d'un fichier de manière mémoire-efficace ;
- utiliser `hmac` pour authentifier des messages ;
- choisir le bon algorithme selon le cas d'usage.

## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :
- les types `bytes` et `str` et l'encodage UTF-8 ;
- la lecture de fichiers en mode binaire ;
- les notions de base de la sécurité (ce notebook les approfondit).

## Plan

1. Qu'est-ce qu'une fonction de hachage ?
2. `hashlib` — calculer des empreintes
3. Algorithmes disponibles
4. Hacher un fichier
5. `hmac` — authentification de messages
6. Comparaison en temps constant
7. BLAKE2 — le couteau suisse
8. Anti-patterns
9. Synthèse
10. Exercices
11. Ressources

---

## 1. Qu'est-ce qu'une fonction de hachage ?

Une fonction de hachage transforme une entrée de **taille arbitraire** en une sortie de **taille fixe** (l'empreinte ou *digest*).

| Propriété | Description |
|---|---|
| **Déterminisme** | Même entrée → même sortie |
| **Rapidité** | Calcul en O(n) de la taille de l'entrée |
| **Résistance aux pré-images** | Impossible de retrouver l'entrée à partir du hash |
| **Résistance aux collisions** | Quasi-impossible de trouver deux entrées avec le même hash |
| **Effet avalanche** | Un bit modifié → hash radicalement différent |

---

## 2. `hashlib` — calculer des empreintes

In [ ]:
import hashlib

In [ ]:
# SHA-256 d'une chaîne
h = hashlib.sha256(b"Hello, World!")
print(f"Digest hex : {h.hexdigest()}")
print(f"Digest bytes : {h.digest()[:8]}... ({h.digest_size} octets)")

`hashlib` attend des **bytes**, pas des `str`. Si vous avez une chaîne, encodez-la :

In [ ]:
message = "Bonjour le monde"
h = hashlib.sha256(message.encode("utf-8"))
print(h.hexdigest())

### Mise à jour incrémentale

In [ ]:
h = hashlib.sha256()
h.update(b"Hello, ")
h.update(b"World!")
print(h.hexdigest())

In [ ]:
# Équivalent en un seul appel
h2 = hashlib.sha256(b"Hello, World!")
print(h2.hexdigest())
print(f"Identiques : {h.hexdigest() == h2.hexdigest()}")

La mise à jour incrémentale est essentielle pour **hacher de gros fichiers** sans tout charger en mémoire.

### Effet avalanche

In [ ]:
h1 = hashlib.sha256(b"Hello").hexdigest()
h2 = hashlib.sha256(b"Hellp").hexdigest()  # un seul caractère change
print(f"Hello : {h1}")
print(f"Hellp : {h2}")
# Les deux hashes sont complètement différents

---

## 3. Algorithmes disponibles

In [ ]:
print("Garantis :", sorted(hashlib.algorithms_guaranteed))

In [ ]:
print("Disponibles :", sorted(hashlib.algorithms_available)[:20], "...")

| Algorithme | Taille | Statut | Usage recommandé |
|---|---|---|---|
| MD5 | 128 bits | **Cassé** | Checksums non-sécurité uniquement |
| SHA-1 | 160 bits | **Cassé** | Éviter |
| SHA-256 | 256 bits | Sûr | Usage général |
| SHA-512 | 512 bits | Sûr | Quand 256 bits ne suffit pas |
| SHA-3-256 | 256 bits | Sûr | Alternative post-quantique |
| BLAKE2b | 1-512 bits | Sûr | Performance + sécurité |
| BLAKE2s | 1-256 bits | Sûr | Performance sur 32 bits |

### MD5 n'est plus sûr

In [ ]:
# MD5 est rapide mais VULNÉRABLE aux collisions
# Ne l'utilisez JAMAIS pour la sécurité
h = hashlib.md5(b"test")
print(f"MD5 : {h.hexdigest()} ({h.digest_size * 8} bits)")

h2 = hashlib.sha256(b"test")
print(f"SHA-256 : {h2.hexdigest()} ({h2.digest_size * 8} bits)")

### Performances comparées

In [ ]:
import timeit

data = b"x" * 1_000_000  # 1 Mo

for algo in ["md5", "sha256", "sha512", "sha3_256", "blake2b"]:
    fn = getattr(hashlib, algo)
    t = timeit.timeit(lambda: fn(data).hexdigest(), number=100)
    print(f"{algo:12s} : {t:.3f}s pour 100 Mo")

---

## 4. Hacher un fichier

Pour les fichiers volumineux, on lit par blocs pour éviter de tout charger en mémoire.

In [ ]:
def hash_fichier(chemin, algorithme="sha256", taille_bloc=65536):
    """Calcule le hash d'un fichier par blocs."""
    h = hashlib.new(algorithme)
    with open(chemin, "rb") as f:
        while True:
            bloc = f.read(taille_bloc)
            if not bloc:
                break
            h.update(bloc)
    return h.hexdigest()

In [ ]:
import tempfile
import os

# Créer un fichier de test
with tempfile.NamedTemporaryFile(delete=False, suffix=".txt") as f:
    tmp_path = f.name
    f.write(b"Contenu du fichier de test\n" * 10_000)

digest = hash_fichier(tmp_path)
print(f"SHA-256 du fichier : {digest}")
os.unlink(tmp_path)

### Python 3.11+ : `hashlib.file_digest()`

In [ ]:
# Depuis Python 3.11, une fonction dédiée existe
import sys

if sys.version_info >= (3, 11):
    with tempfile.NamedTemporaryFile(delete=False) as f:
        tmp_path = f.name
        f.write(b"test data\n" * 1000)

    with open(tmp_path, "rb") as f:
        digest = hashlib.file_digest(f, "sha256")
    print(f"file_digest : {digest.hexdigest()}")
    os.unlink(tmp_path)
else:
    print(f"Python {sys.version_info[:2]} — file_digest nécessite 3.11+")

### Vérifier l'intégrité d'un téléchargement

In [ ]:
def verifier_integrite(chemin, hash_attendu, algorithme="sha256"):
    """Vérifie qu'un fichier correspond au hash attendu."""
    hash_calcule = hash_fichier(chemin, algorithme)
    if hash_calcule == hash_attendu:
        print(f"Intégrité vérifiée : {chemin}")
        return True
    else:
        print(f"ERREUR : hash attendu {hash_attendu[:16]}..., obtenu {hash_calcule[:16]}...")
        return False

---

## 5. `hmac` — authentification de messages

**HMAC** (Hash-based Message Authentication Code) combine une **clé secrète** et un hash pour authentifier un message. Contrairement au hash simple, HMAC prouve que le message a été créé par quelqu'un qui connaît la clé.

In [ ]:
import hmac

In [ ]:
cle_secrete = b"ma-cle-super-secrete"
message = b"Transfert de 1000 euros vers FR76..."

signature = hmac.new(cle_secrete, message, hashlib.sha256).hexdigest()
print(f"HMAC-SHA256 : {signature}")

### Vérifier un HMAC

In [ ]:
def verifier_hmac(cle, message, signature_recue, algorithme=hashlib.sha256):
    """Vérifie un HMAC de manière sécurisée (temps constant)."""
    signature_calculee = hmac.new(cle, message, algorithme).hexdigest()
    return hmac.compare_digest(signature_calculee, signature_recue)

# Vérification correcte
print(verifier_hmac(cle_secrete, message, signature))

# Vérification avec message modifié
message_altere = b"Transfert de 9999 euros vers FR76..."
print(verifier_hmac(cle_secrete, message_altere, signature))

### Pourquoi `hmac.compare_digest()` et pas `==` ?

L'opérateur `==` sur les chaînes effectue une comparaison **caractère par caractère** et s'arrête au premier caractère différent. Un attaquant peut mesurer le temps de la comparaison pour deviner le HMAC correct octet par octet (**timing attack**).

`hmac.compare_digest()` compare en **temps constant** : elle prend toujours le même temps, quelle que soit la position de la différence.

### Cas d'usage du HMAC

| Cas d'usage | Description |
|---|---|
| Webhooks | Vérifier que la requête vient bien du service attendu |
| Tokens signés | JWT (JSON Web Tokens) utilise HMAC-SHA256 |
| API signing | Signer les requêtes API (AWS Signature V4) |
| Intégrité + authenticité | Prouver que le message n'a pas été modifié |

### Signer un webhook

In [ ]:
import json as json_mod

def signer_payload(cle: bytes, payload: dict) -> str:
    """Signe un payload JSON et retourne la signature hex."""
    body = json_mod.dumps(payload, sort_keys=True).encode("utf-8")
    return hmac.new(cle, body, hashlib.sha256).hexdigest()

def verifier_webhook(cle: bytes, payload: dict, signature: str) -> bool:
    """Vérifie la signature d'un webhook."""
    attendue = signer_payload(cle, payload)
    return hmac.compare_digest(attendue, signature)

# Simulation
cle = b"webhook-secret-key-12345"
payload = {"event": "payment", "amount": 42.0, "user": "alice"}
sig = signer_payload(cle, payload)
print(f"Signature : {sig}")
print(f"Valide : {verifier_webhook(cle, payload, sig)}")

# Payload altéré
payload_altere = {"event": "payment", "amount": 9999.0, "user": "alice"}
print(f"Altéré : {verifier_webhook(cle, payload_altere, sig)}")

---

## 6. Comparaison en temps constant

In [ ]:
# hmac.compare_digest fonctionne aussi avec des bytes
a = b"signature_correcte"
b_ok = b"signature_correcte"
b_ko = b"signature_fausse!!"

print(hmac.compare_digest(a, b_ok))
print(hmac.compare_digest(a, b_ko))

**Utilisez toujours `hmac.compare_digest()` pour comparer des secrets**, jamais `==`.

---

## 7. BLAKE2 — le couteau suisse

BLAKE2 est une famille de fonctions de hachage **rapides et sûres**, intégrées à Python depuis 3.6.

In [ ]:
# BLAKE2b — optimisé pour les CPU 64 bits
h = hashlib.blake2b(b"Hello", digest_size=32)
print(f"BLAKE2b-256 : {h.hexdigest()}")

In [ ]:
# Taille de digest personnalisable (1 à 64 octets)
h16 = hashlib.blake2b(b"Hello", digest_size=16)
print(f"BLAKE2b-128 : {h16.hexdigest()} ({len(h16.digest())} octets)")

### BLAKE2 avec clé (keyed hashing) — alternative à HMAC

In [ ]:
# BLAKE2 supporte nativement le keyed hashing
cle = b"ma-cle-secrete-blake2"
h = hashlib.blake2b(b"message important", key=cle, digest_size=32)
print(f"BLAKE2b keyed : {h.hexdigest()}")

Le keyed hashing BLAKE2 est **plus rapide** que HMAC-SHA256 car il intègre la clé directement dans l'algorithme, sans le double hashing de HMAC.

### BLAKE2 pour la dérivation d'identifiants

In [ ]:
# Générer un identifiant court et unique à partir d'un email
def email_to_id(email: str) -> str:
    return hashlib.blake2b(
        email.encode(), digest_size=8
    ).hexdigest()

print(email_to_id("alice@example.com"))
print(email_to_id("bob@example.com"))

---

## 8. Anti-patterns

### Ne JAMAIS utiliser un hash simple pour les mots de passe

```python
# DANGEREUX — ne faites jamais cela !
hashlib.sha256(password.encode()).hexdigest()
```

Un hash simple est vulnérable aux :
- **Rainbow tables** : tables précalculées de hash → mot de passe ;
- **Brute force** : SHA-256 est rapide (~1 milliard de hash/s sur GPU).

Utilisez `argon2`, `bcrypt` ou `scrypt` pour les mots de passe (voir notebook suivant).

### Ne pas utiliser MD5 ou SHA-1 pour la sécurité

MD5 et SHA-1 ont des **collisions connues**. Un attaquant peut créer deux fichiers différents avec le même hash.

### Ne pas comparer des hashes avec `==`

Toujours utiliser `hmac.compare_digest()` pour éviter les timing attacks.

---

## 9. Synthèse

| Outil | Usage |
|---|---|
| `hashlib.sha256()` | Empreinte standard (intégrité) |
| `hashlib.blake2b()` | Empreinte rapide + sûre |
| `hashlib.file_digest()` | Hash de fichier (Python 3.11+) |
| `hmac.new(clé, msg, algo)` | Authentification de messages |
| `hmac.compare_digest(a, b)` | Comparaison en temps constant |
| `hashlib.blake2b(msg, key=k)` | Keyed hashing (alternative HMAC) |

**Règles à retenir :**
- SHA-256 ou BLAKE2 pour le hachage général ; jamais MD5/SHA-1 pour la sécurité.
- HMAC pour prouver à la fois l'intégrité **et** l'authenticité.
- Toujours `hmac.compare_digest()`, jamais `==` pour comparer des secrets.
- **Jamais** de hash simple pour les mots de passe (voir notebook suivant).

---

## 10. Exercices

### Exercice 1 — Hash d'un fichier *(facile)*

Écrire une fonction `empreinte(chemin: str, algo: str = "sha256") -> str` qui retourne le hash hexadécimal d'un fichier, lu par blocs de 64 Ko. Testez-la sur un fichier temporaire.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Hashing_et_hmac", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import hashlib
import tempfile
import os

def empreinte(chemin: str, algo: str = "sha256") -> str:
    h = hashlib.new(algo)
    with open(chemin, "rb") as f:
        while bloc := f.read(65536):
            h.update(bloc)
    return h.hexdigest()

with tempfile.NamedTemporaryFile(delete=False) as f:
    f.write(b"Donnees de test\n" * 1000)
    tmp = f.name

print(f"SHA-256 : {empreinte(tmp)}")
print(f"BLAKE2  : {empreinte(tmp, 'blake2b')}")
os.unlink(tmp)
```

</details>

### Exercice 2 — Détecter les fichiers identiques *(moyen)*

Écrire une fonction `trouver_doublons(dossier: str) -> dict[str, list[str]]` qui parcourt un dossier et regroupe les fichiers ayant le même contenu (même hash SHA-256). Testez avec des fichiers temporaires.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Hashing_et_hmac", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import hashlib
import os
import tempfile
from collections import defaultdict

def hash_fichier(chemin):
    h = hashlib.sha256()
    with open(chemin, "rb") as f:
        while bloc := f.read(65536):
            h.update(bloc)
    return h.hexdigest()

def trouver_doublons(dossier):
    par_hash = defaultdict(list)
    for entry in os.scandir(dossier):
        if entry.is_file():
            h = hash_fichier(entry.path)
            par_hash[h].append(entry.name)
    return {h: noms for h, noms in par_hash.items() if len(noms) > 1}

# Test
with tempfile.TemporaryDirectory() as d:
    for name, content in [("a.txt", b"hello"), ("b.txt", b"hello"),
                          ("c.txt", b"world"), ("d.txt", b"world"),
                          ("e.txt", b"unique")]:
        with open(os.path.join(d, name), "wb") as f:
            f.write(content)
    doublons = trouver_doublons(d)
    for h, fichiers in doublons.items():
        print(f"Hash {h[:12]}... : {fichiers}")
```

</details>

### Exercice 3 — Middleware HMAC *(moyen)*

Implémenter deux fonctions :
- `signer_requete(cle, methode, chemin, body, timestamp)` → signature hex
- `verifier_requete(cle, methode, chemin, body, timestamp, signature)` → bool

Le message signé doit être `f"{methode}\n{chemin}\n{timestamp}\n{body}"`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Hashing_et_hmac", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import hmac
import hashlib

def _construire_message(methode, chemin, body, timestamp):
    return f"{methode}\n{chemin}\n{timestamp}\n{body}".encode("utf-8")

def signer_requete(cle, methode, chemin, body, timestamp):
    msg = _construire_message(methode, chemin, body, timestamp)
    return hmac.new(cle, msg, hashlib.sha256).hexdigest()

def verifier_requete(cle, methode, chemin, body, timestamp, signature):
    attendue = signer_requete(cle, methode, chemin, body, timestamp)
    return hmac.compare_digest(attendue, signature)

cle = b"api-secret-key"
sig = signer_requete(cle, "POST", "/api/paiement", '{"montant": 100}', "2026-04-14T10:00:00Z")
print(f"Signature : {sig}")
print(f"Valide    : {verifier_requete(cle, 'POST', '/api/paiement', '{"montant": 100}', '2026-04-14T10:00:00Z', sig)}")
print(f"Altéré    : {verifier_requete(cle, 'POST', '/api/paiement', '{"montant": 999}', '2026-04-14T10:00:00Z', sig)}")
```

</details>

### Exercice 4 — Cache avec invalidation par hash *(difficile)*

Implémenter un système de cache fichier où chaque entrée est invalidée si le fichier source a changé (détecté par son hash). L'interface doit être :

```python
cache = HashCache(dossier_cache="/tmp/hcache")
result = cache.get_or_compute("data.csv", lambda path: traiter(path))
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Hashing_et_hmac", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import hashlib
import json
import os
import tempfile

class HashCache:
    def __init__(self, dossier_cache):
        self.dossier = dossier_cache
        os.makedirs(dossier_cache, exist_ok=True)

    def _hash_fichier(self, chemin):
        h = hashlib.sha256()
        with open(chemin, "rb") as f:
            while bloc := f.read(65536):
                h.update(bloc)
        return h.hexdigest()

    def _chemin_cache(self, chemin_source):
        nom = hashlib.sha256(chemin_source.encode()).hexdigest()[:16]
        return os.path.join(self.dossier, f"{nom}.json")

    def get_or_compute(self, chemin_source, fn):
        cache_path = self._chemin_cache(chemin_source)
        hash_actuel = self._hash_fichier(chemin_source)

        if os.path.exists(cache_path):
            with open(cache_path) as f:
                entry = json.load(f)
            if entry["hash"] == hash_actuel:
                return entry["result"]

        result = fn(chemin_source)
        with open(cache_path, "w") as f:
            json.dump({"hash": hash_actuel, "result": result}, f)
        return result

# Test
with tempfile.TemporaryDirectory() as cache_dir:
    with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", delete=False) as f:
        f.write("a,b\n1,2\n3,4\n")
        data_path = f.name

    cache = HashCache(cache_dir)
    r1 = cache.get_or_compute(data_path, lambda p: len(open(p).readlines()))
    print(f"Résultat (calcul) : {r1}")
    r2 = cache.get_or_compute(data_path, lambda p: len(open(p).readlines()))
    print(f"Résultat (cache)  : {r2}")
    os.unlink(data_path)
```

</details>

---

## 11. Ressources

- [Module `hashlib` — documentation officielle](https://docs.python.org/3/library/hashlib.html)
- [Module `hmac` — documentation officielle](https://docs.python.org/3/library/hmac.html)
- [BLAKE2 — site officiel](https://www.blake2.net/)
- [OWASP — Password Storage Cheat Sheet](https://cheatsheetseries.owasp.org/cheatsheets/Password_Storage_Cheat_Sheet.html)